In [18]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rapidfuzz import fuzz
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
import joblib

df_bank = pd.read_csv("../train_data/bank_statements.csv")
df_gl = pd.read_csv("../train_data/gl_ledger.csv")
df_invoices = pd.read_csv("../train_data/invoices.csv")
df_truth = pd.read_csv("../train_data/ground_truth.csv")

In [19]:
# Convert to datetime vectors once
df_bank['date'] = pd.to_datetime(df_bank['date'])
df_gl['date'] = pd.to_datetime(df_gl['date'])

# Feature: Count occurrences of identical bank transactions (duplicates)
df_bank['bank_dup_count'] = df_bank.groupby(['amount', 'counterparty', 'date'])['amount'].transform('count')

# Perform an in-memory cross join (pushes O(N*M) loop to C)
df_cross = df_bank.merge(df_gl, how='cross', suffixes=('_b', '_g'))

# Vectorized condition checks
date_diffs = (df_cross['date_b'] - df_cross['date_g']).dt.days.abs()
amt_diff_pcts = (df_cross['amount_b'] - df_cross['amount_g']).abs() / np.maximum(df_cross['amount_b'], 0.01)

# Apply blocking filters simultaneously
mask = (date_diffs <= 3) & (amt_diff_pcts <= 0.05)
df_candidates = df_cross[mask].copy()

# Rename columns to match the expected downstream schema
df_candidates = df_candidates.rename(columns={
    'amount_b': 'b_amount',
    'amount_g': 'g_amount',
    'date_b': 'b_date',
    'date_g': 'g_date',
    'description': 'b_desc',
    'counterparty': 'b_counterparty',
    'memo': 'g_memo',
    'vendor_id': 'g_vendor_id'
})

# Keep only necessary columns (now including bank_dup_count)
df_candidates = df_candidates[[
    'txn_id', 'entry_id', 'b_amount', 'g_amount', 
    'b_date', 'g_date', 'b_desc', 'b_counterparty', 
    'g_memo', 'g_vendor_id', 'bank_dup_count'
]]

In [20]:
# 1. Reset index to prevent Pandas from misaligning rows during math
df_candidates = df_candidates.reset_index(drop=True)
df_features = df_candidates[['txn_id', 'entry_id']].copy()

# 2. Extract basic numeric features
df_features['amount_diff'] = (df_candidates['b_amount'] - df_candidates['g_amount']).abs()
df_features['amount_ratio'] = np.minimum(df_candidates['b_amount'], df_candidates['g_amount']) / np.maximum(df_candidates['b_amount'], df_candidates['g_amount'])
df_features['date_diff_days'] = (pd.to_datetime(df_candidates['b_date']) - pd.to_datetime(df_candidates['g_date'])).dt.days.abs()
df_features['direction_match'] = 1
df_features['bank_dup_count'] = df_candidates['bank_dup_count']

# 3. Vectorized TF-IDF Cosine Similarity
all_text = pd.concat([df_bank['description'], df_gl['memo']]).unique()
tfidf = TfidfVectorizer().fit(all_text)
b_vecs = tfidf.transform(df_candidates['b_desc'])
g_vecs = tfidf.transform(df_candidates['g_memo'])
df_features['desc_cosine'] = np.asarray(b_vecs.multiply(g_vecs).sum(axis=1)).flatten()

# 4. Fuzzy Matching (List comprehension is fastest here for RapidFuzz)
df_features['counterparty_fuzzy'] = [
    fuzz.ratio(str(b), str(g)) / 100.0 
    for b, g in zip(df_candidates['b_counterparty'], df_candidates['g_vendor_id'])
]

# 5. Invoice Logic
merged_inv = df_candidates[['txn_id', 'entry_id', 'g_vendor_id', 'b_amount']].merge(
    df_invoices[['vendor_id', 'amount']], 
    left_on='g_vendor_id', 
    right_on='vendor_id', 
    how='left'
)
# Drop duplicates in case a vendor has multiple invoices for the exact same amount
valid_inv = merged_inv[(merged_inv['amount'] - merged_inv['b_amount']).abs() <= 2.0].drop_duplicates(subset=['txn_id', 'entry_id'])

df_features = df_features.merge(
    valid_inv[['txn_id', 'entry_id', 'amount']], 
    on=['txn_id', 'entry_id'], 
    how='left'
)

# Because indices now perfectly align, this math will work instantly
df_features['has_invoice'] = df_features['amount'].notna().astype(int)
df_features['invoice_amount_diff'] = np.where(
    df_features['has_invoice'] == 1, 
    (df_candidates['b_amount'] - df_features['amount']).abs(), 
    -1.0
)
df_features = df_features.drop(columns=['amount'])

# 6. Attach Ground Truth Labels
df_features = df_features.merge(
    df_truth[['txn_id', 'entry_id', 'match_label']],
    on=['txn_id', 'entry_id'],
    how='left'
)
df_features['match_label'] = df_features['match_label'].fillna(0).astype(int)

In [21]:
feature_cols = [
    'amount_diff', 'amount_ratio', 'date_diff_days', 
    'desc_cosine', 'counterparty_fuzzy', 'direction_match', 
    'has_invoice', 'invoice_amount_diff', 'bank_dup_count'
]

X = df_features[feature_cols]
y = df_features['match_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.99, random_state=42, stratify=y
)

test_indices = X_test.index
df_test = df_features.loc[test_indices]
df_test.to_csv('./test_data.csv', index=False)

clf = GradientBoostingClassifier(
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=3, 
    random_state=42
)
clf.fit(X_train, y_train)

joblib.dump(clf, './reconciliation_model.pkl')

['./reconciliation_model.pkl']

In [22]:
y_probs = clf.predict_proba(X_test)[:, 1]
THRESHOLD = 0.75
y_pred_thresholded = (y_probs >= THRESHOLD).astype(int)

precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred_thresholded, average='binary', zero_division=0)
auc_roc = roc_auc_score(y_test, y_probs)

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"AUC-ROC:   {auc_roc:.4f}")

feature_importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': clf.feature_importances_
}).sort_values(by='Importance', ascending=False)
print(feature_importance_df)

Precision: 0.9711
Recall:    0.9648
F1 Score:  0.9679
AUC-ROC:   0.9998
               Feature    Importance
0          amount_diff  9.265683e-01
3          desc_cosine  5.599628e-02
4   counterparty_fuzzy  1.569380e-02
1         amount_ratio  1.741615e-03
2       date_diff_days  0.000000e+00
5      direction_match  0.000000e+00
6          has_invoice  0.000000e+00
8       bank_dup_count  0.000000e+00
7  invoice_amount_diff -3.040982e-17


In [23]:
df_features['match_prob'] = clf.predict_proba(df_features[feature_cols])[:, 1]

# Filter down to just the pairs that passed the probability threshold
df_high_conf = df_features[df_features['match_prob'] >= 0.90].copy()

# Count how many GL entries passed the threshold for each bank transaction
match_counts = df_high_conf['txn_id'].value_counts()

# A "safe" match is one where EXACTLY ONE candidate pair passed the threshold
safe_matched_txn_ids = match_counts[match_counts == 1].index

# Route the safe matches to the final ledger
df_matches = df_high_conf[df_high_conf['txn_id'].isin(safe_matched_txn_ids)].copy()

# Route everything else (low confidence OR conflicting multi-matches) to the agent
unmatched_bank_txns = df_bank[~df_bank['txn_id'].isin(safe_matched_txn_ids)].copy()

df_matches.to_csv("../train_data/confirmed_matches.csv", index=False)
unmatched_bank_txns.to_csv("../train_data/exceptions_for_agent.csv", index=False)

print(f"Auto-Matched: {len(df_matches)}")
print(f"Exceptions (low confidence + conflicts): {len(unmatched_bank_txns)}")

Auto-Matched: 4183
Exceptions (low confidence + conflicts): 817


In [24]:
from sklearn.metrics import classification_report
from sklearn.dummy import DummyClassifier
import pandas as pd

# 1. Baseline — prove the ML model beats a naive "Reject All" strategy
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
print("=== Dummy Baseline (Predict Exception for Everything) ===")
print(classification_report(y_test, dummy.predict(X_test), zero_division=0))

# 2. Evaluate the actual model with the strict threshold
print("\n=== XGBoost Model (Threshold: 0.93) ===")
y_pred = (clf.predict_proba(X_test)[:, 1] >= 0.93).astype(int)
report = classification_report(y_test, y_pred, target_names=['Exception (0)', 'Match (1)'])
print(report)
# -> Look directly at the 'Match (1)' row to see your true Precision and Recall.

# 3. Check for Data Leakage & Drivers
importances = pd.Series(
    clf.feature_importances_, 
    index=feature_cols
).sort_values(ascending=False)

print("\n=== Top 5 Feature Drivers ===")
print(importances.head(5))
# If amount_ratio and amount_diff are at the top, the model learned finance.
# (Since feature_cols excluded txn_id/entry_id, leakage is safely prevented).

=== Dummy Baseline (Predict Exception for Everything) ===
              precision    recall  f1-score   support

           0       0.99      1.00      0.99    320321
           1       0.00      0.00      0.00      4208

    accuracy                           0.99    324529
   macro avg       0.49      0.50      0.50    324529
weighted avg       0.97      0.99      0.98    324529


=== XGBoost Model (Threshold: 0.93) ===
               precision    recall  f1-score   support

Exception (0)       1.00      1.00      1.00    320321
    Match (1)       0.97      0.96      0.97      4208

     accuracy                           1.00    324529
    macro avg       0.99      0.98      0.98    324529
 weighted avg       1.00      1.00      1.00    324529


=== Top 5 Feature Drivers ===
amount_diff           0.926568
desc_cosine           0.055996
counterparty_fuzzy    0.015694
amount_ratio          0.001742
date_diff_days        0.000000
dtype: float64
